In [ ]:
#| default_exp size

In [ ]:
#| include: false
from nbdev.showdoc import *

In [ ]:
#| export
from __future__ import annotations

from fasterbench.core import _bytes_to_mib
import torch
import io
from dataclasses import dataclass, asdict

In [ ]:
#| export
def get_model_size(model: torch.nn.Module) -> int:  # model to measure
    """Return the on-disk size (bytes) of the serialized model."""
    buf = io.BytesIO()
    try:
        model.save(buf)
    except Exception:
        torch.save(model.state_dict(), buf)
    return buf.getbuffer().nbytes


#| export
def get_num_parameters(
    model: torch.nn.Module,       # model to count parameters
    trainable_only: bool = True,  # if True, only count trainable parameters
) -> int:
    """Count the number of (optionally trainable) parameters."""
    if trainable_only:
        return sum(p.numel() for p in model.parameters() if p.requires_grad)
    return sum(p.numel() for p in model.parameters())


#| export
@dataclass(slots=True)
class SizeMetrics:
    """Model size metrics: disk size and parameter count."""
    disk_bytes: int
    size_mib: float
    num_params: int

    def as_dict(self) -> dict[str, float]:
        return asdict(self)


#| export
def compute_size(
    model: torch.nn.Module,             # model to measure
    *,
    params_count: int | None = None,    # pre-computed parameter count (avoids recount)
) -> SizeMetrics:
    """Compute size metrics for a model."""
    disk = get_model_size(model)
    params = params_count if params_count is not None else get_num_parameters(model)
    return SizeMetrics(disk_bytes=disk, size_mib=_bytes_to_mib(disk), num_params=params)

In [ ]:
show_doc(SizeMetrics)

In [ ]:
show_doc(compute_size)

In [ ]:
show_doc(get_model_size)

In [ ]:
show_doc(get_num_parameters)

In [ ]:
#| hide
from fastcore.test import *

import torch.nn as nn
_m = nn.Linear(10, 5)
_s = compute_size(_m)
assert isinstance(_s, SizeMetrics)
assert _s.num_params > 0
assert _s.size_mib > 0
test_eq(get_num_parameters(_m), 55)  # 10*5 + 5 bias

---

## See Also

- [Benchmark](../analysis/benchmark.html) - Unified benchmarking with `benchmark()`
- [Compute](compute.html) - MACs and operation counts
- [Profiling](../analysis/profiling.html) - Per-layer size analysis